# 实践项目 03：脑膜瘤 H&E 形态代理分类

本 Notebook 使用课程准备的真实脑膜瘤 H&E 图块数据，完成数据核对、颜色统计基线、小型 CNN、原图级测试和染色扰动比较。三档标签由核密度代理分数生成，只用于教学分类。

Kaggle 是本项目的首选实践入口。打开公开 Notebook 后，点击“复制并编辑”保存到自己的账户，再按单元格顺序运行。下载 Notebook 到电脑运行是补充方式。

代码中用整行注释标出了需要填写的位置。先阅读当前单元格的输入、处理和输出，再修改标记区域。合理利用 AI 工具理解问题、学习知识并尝试给出适当的解决方案。

## 任务总览

1. 找到固定的课程 NPZ，并核对图块 shape、标签、原图编号和数据划分。
2. 输出三类标签数量，并比较颜色统计基线。
3. 补全小型 CNN 的特征提取和分类层。
4. 完成训练更新、验证集选模和测试评价。
5. 查看混淆矩阵、错误图块和染色变化结果。

## 需要保存的结果

`task3_data_visualization.png`、`task3_training_curve.png`、`task3_prediction_visualization.png`、`task3_pytorch_result.json`。

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working'); OUT.mkdir(exist_ok=True)
DATA_PATH = None  # 可在本地填写课程 NPZ 路径；Kaggle 自动查找固定文件名
candidates = sorted(INPUT.rglob('meningioma_public_morphology_tiles.npz'))
if DATA_PATH is None and candidates:
    DATA_PATH = candidates[0]
assert DATA_PATH is not None, '请挂载包含 meningioma_public_morphology_tiles.npz 的课程数据集。'
data = np.load(DATA_PATH, allow_pickle=True)
required = {'images','labels','source_image_ids','split'}
assert required.issubset(data.files), required - set(data.files)
images = data['images']; labels = data['labels']; source_image_ids = data['source_image_ids'].astype(str); split = data['split'].astype(str)
assert images.shape[1:] == (128, 128, 3)
print('data:', images.shape, 'splits:', {k: int((split == k).sum()) for k in np.unique(split)})

## 任务 1：完成数据核对

输入是 `images`、`labels`、`source_image_ids` 和 `split`。请输出每个 split 的图块数、原图编号和三类标签数量。

In [ ]:
# ===== 项目03·任务1·学生填写区（开始） =====
# TODO：根据 images、labels、source_image_ids 和 split 生成 summary。
summary = None
# ===== 项目03·任务1·学生填写区（结束） =====
print(summary)

## 任务 2：建立颜色统计基线

标签来自核密度代理分数，因此先用 RGB 均值和标准差建立一个简单基线。

In [ ]:
def color_features(batch):
    z = batch.astype(np.float32) / 255.0
    return np.c_[z.mean((1, 2)), z.std((1, 2))]
tr = np.where(split == 'train')[0]; te = np.where(split == 'test')[0]
X = color_features(images)
baseline = LogisticRegression(max_iter=1200, class_weight='balanced')
baseline.fit(X[tr], labels[tr])
base_pred = baseline.predict(X[te])
base_f1 = f1_score(labels[te], base_pred, average='macro')
print('baseline macro F1:', base_f1)

## 任务 3：补全小型 CNN

输入为 `[batch, 3, 128, 128]`，输出为三个类别的 logits。

In [ ]:
class PatchDataset(Dataset):
    def __init__(self, indices, augment=False): self.indices = np.asarray(indices); self.augment = augment
    def __len__(self): return len(self.indices)
    def __getitem__(self, k):
        i = int(self.indices[k]); x = images[i].astype(np.float32) / 255.0
        if self.augment and random.random() < 0.5: x = np.fliplr(x).copy()
        return torch.from_numpy(x.transpose(2, 0, 1)), torch.tensor(labels[i]), i

class PatchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # ===== 项目03·任务3·学生填写区（开始） =====
        # TODO：补全三组卷积模块、自适应平均池化和三分类层。
        self.features = None
        self.classifier = None
        # ===== 项目03·任务3·学生填写区（结束） =====
    def forward(self, x): return self.classifier(self.features(x).flatten(1))

model = PatchCNN().to(DEVICE)
print(model)

## 任务 4：训练、验证和测试

验证集用于选择模型，测试集只在设置确定后使用。

In [ ]:
train_idx = np.where(split == 'train')[0]; val_idx = np.where(split == 'validation')[0]; test_idx = np.where(split == 'test')[0]
train_loader = DataLoader(PatchDataset(train_idx, True), 32, shuffle=True)
val_loader = DataLoader(PatchDataset(val_idx), 64, shuffle=False)
test_loader = DataLoader(PatchDataset(test_idx), 64, shuffle=False)
loss_fn = nn.CrossEntropyLoss(); optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ===== 项目03·任务4·学生填写区（开始） =====
# TODO：完成清零梯度、前向计算、损失、反向传播、更新和验证选模。
# ===== 项目03·任务4·学生填写区（结束） =====
print('请先完成训练循环，再运行后续评价。')

## 任务 5：查看错误图块和染色变化

完成模型后，输出混淆矩阵、错误图块和染色扰动后的宏平均 F1。

In [ ]:
# ===== 项目03·任务5·学生填写区（开始） =====
# TODO：完成测试评价、错误图块可视化、染色扰动和 JSON 保存。
# ===== 项目03·任务5·学生填写区（结束） =====
print('输出文件应写入', OUT)